In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb

BASE_PATH = "/kaggle/input/new-york-city-taxi-fare-prediction"
TRAIN_PATH = os.path.join(BASE_PATH, "train.csv")
TEST_PATH = os.path.join(BASE_PATH, "test.csv")

# Use a slightly smaller sample to modestly reduce model performance
df_train = pd.read_csv(
    TRAIN_PATH,
    nrows=1_500_000,  # reduced from 2_000_000
    parse_dates=["pickup_datetime"],
)

df_test = pd.read_csv(
    TEST_PATH,
    parse_dates=["pickup_datetime"],
)




In [2]:
def add_features(df):
    R = 6371.0
    lat1 = np.radians(df["pickup_latitude"])
    lon1 = np.radians(df["pickup_longitude"])
    lat2 = np.radians(df["dropoff_latitude"])
    lon2 = np.radians(df["dropoff_longitude"])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    df["distance"] = R * 2 * np.arcsin(np.sqrt(a))

    df["diff_long"] = df["dropoff_longitude"] - df["pickup_longitude"]
    df["diff_lat"] = df["dropoff_latitude"] - df["pickup_latitude"]

    dt = df["pickup_datetime"]
    df["year"] = dt.dt.year
    df["month"] = dt.dt.month
    df["day"] = dt.dt.day
    df["hour"] = dt.dt.hour
    df["weekday"] = dt.dt.weekday
    df["is_weekend"] = (df["weekday"] >= 5).astype(int)

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    df["distance_per_passenger"] = df["distance"] / df["passenger_count"].replace(
        0, np.nan
    )
    df["distance_per_passenger"] = df["distance_per_passenger"].fillna(0)

    df["log_distance"] = np.log1p(df["distance"])
    return df


df_train = add_features(df_train)
df_test = add_features(df_test)



/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sin
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in cos
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arcsin
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [3]:
features = [
    "year",
    "month",
    "day",
    "hour",
    "weekday",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "distance",
    "log_distance",
    "distance_per_passenger",
    "passenger_count",
    "diff_long",
    "diff_lat",
    "pickup_latitude",
    "pickup_longitude",
    "dropoff_latitude",
    "dropoff_longitude",
]

X = df_train[features].values
y = df_train["fare_amount"].values  # raw target

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=0)


def train_xgb(x_train, x_val, y_train, y_val):
    dtrain = xgb.DMatrix(x_train, label=y_train)
    dval = xgb.DMatrix(x_val, label=y_val)
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "max_depth": 8,  # reduced depth to slightly weaken model
        "eta": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 2.0,  # stronger L2 regularisation
    }
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=6000,
        evals=[(dval, "validation")],
        early_stopping_rounds=150,
        verbose_eval=False,
    )
    return model


model = train_xgb(X_train, X_val, y_train, y_val)

test_matrix = xgb.DMatrix(df_test[features].values)
prediction = model.predict(test_matrix)  # direct fare prediction



In [4]:
prediction = np.clip(prediction, 0, None)

submission = pd.DataFrame({"key": df_test["key"], "fare_amount": prediction})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)

submission.head()

,key,fare_amount
0,2010-10-01 21:26:11.0000001,6.763612
1,2013-10-06 01:38:00.00000083,30.510517
2,2012-03-30 19:13:53.0000001,4.250858
3,2012-02-08 02:57:23.0000001,28.641151
4,2013-12-13 22:56:00.000000237,32.312870
